# Penguin-VL Inference Recipes

This notebook mirrors the cases in `inference/example_penguinvl.py` and presents them in a notebook-friendly format that is easier to read on GitHub after execution.

## What This Notebook Covers

1. Generate Python code from a screenshot of an algorithm problem.
2. Parse a newspaper page with OCR-style prompting.
3. Produce creative writing from an artwork.
4. Convert a long table image into Markdown.
5. Run a multi-round chart-understanding conversation.
6. Run a multi-round video-understanding conversation.
7. Mix video and image inputs in a single prompt.
8. Compare with a plain-text-only prompt.

## Before You Run It

- Use the same Python environment that you use for `python inference/example_penguinvl.py`.
- `transformers==4.51.3` is recommended by the repo.
- Set `MODEL_PATH` below to a Hugging Face model ID or a local checkpoint.
- After running the notebook, save it so GitHub can render the outputs inline.


## General: Import Libraries and Locate the Repo

This cell keeps the notebook portable. It searches upward from the current working directory until it finds `inference/example_penguinvl.py`.


In [ ]:
import os
import html
import base64
import mimetypes
import sys
import contextlib
import warnings
from pathlib import Path

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('TRANSFORMERS_VERBOSITY', 'error')

warnings.filterwarnings('ignore')

import torch
import markdown2
from IPython.display import HTML, Markdown, display
from transformers import AutoModelForCausalLM, AutoProcessor


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'inference' / 'example_penguinvl.py').exists():
            return path
    raise FileNotFoundError('Could not locate the Penguin-VL repo root from the current working directory.')


@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, 'w') as devnull:
        stdout_fd = os.dup(1)
        stderr_fd = os.dup(2)
        try:
            with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
                os.dup2(devnull.fileno(), 1)
                os.dup2(devnull.fileno(), 2)
                yield
        finally:
            os.dup2(stdout_fd, 1)
            os.close(stdout_fd)
            os.dup2(stderr_fd, 2)
            os.close(stderr_fd)


REPO_ROOT = find_repo_root(Path.cwd())
ASSETS_DIR = REPO_ROOT / 'assets' / 'inputs'
SCRIPT_PATH = REPO_ROOT / 'inference' / 'example_penguinvl.py'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Located Penguin-VL repo assets successfully.')
print('Script target: inference/example_penguinvl.py')
print('Assets directory: assets/inputs')


## General: Load Model and Processor

This cell intentionally matches the loading logic in `inference/example_penguinvl.py` so the notebook stays aligned with the script.


In [ ]:
MODEL_PATH = os.environ.get('PENGUIN_VL_MODEL_PATH', 'tencent/Penguin-VL-8B')
MODEL_LABEL = Path(MODEL_PATH).name if Path(MODEL_PATH).is_absolute() else MODEL_PATH
print('Using MODEL_PATH:', MODEL_LABEL)

with suppress_output():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
        device_map={'': 'cuda:0'},
        torch_dtype=torch.bfloat16,
        attn_implementation='flash_attention_2',
    )
    processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)

display(Markdown('_Model and processor loaded._'))


## General: Notebook Helpers

The notebook rendering utilities are imported from `inference/notebooks/penguin_notebook_helpers.py`. The code cell below stays intentionally short so readers can focus on the examples instead of the UI plumbing.


In [ ]:
from inference.notebooks.penguin_notebook_helpers import install_notebook_helpers

install_notebook_helpers(
    namespace=globals(),
    model=model,
    processor=processor,
    repo_root=REPO_ROOT,
    assets_dir=ASSETS_DIR,
    suppress_output=suppress_output,
)

display(Markdown('_Notebook helper utilities loaded._'))


## Section 1: Single Image Understanding

This section mirrors the single-image examples in the Python script. Each case uses one image plus one user prompt.


In [ ]:
leetcode_response = run_single_image_case(
    title='Example 1: Code Generation from an Algorithm Screenshot',
    image_name='leetcode.png',
    question='please think this problem step by step and give the python code solution',
)


In [ ]:
ocr_response = run_single_image_case(
    title='Example 2: OCR-Style Parsing from a Newspaper Page',
    image_name='newspaper.png',
    question='please output the text in the image',
)


In [ ]:
poem_response = run_single_image_case(
    title='Example 3: Creative Writing from an Artwork',
    image_name='horse_poet.png',
    question='Write a short poem inspired by this image. Capture the relationship between the man and the horse, as well as the traditional, historical atmosphere of the painting.',
)


In [ ]:
table_response = run_single_image_case(
    title='Example 4: Long Table Parsing to Markdown',
    image_name='2b_table_result.png',
    question='please output the table content in markdown format.',
)


## Section 2: Multi-Round Chart Understanding

This case demonstrates how to carry conversation history forward. The image is only attached in round 1; later rounds rely on the stored assistant context.


In [ ]:
chart_answers = run_multiround_image_case(
    title='Example 5: Multi-Round Chart Understanding',
    image_name='chart_understanding.png',
    questions=[
        "Look at the 'Nonmetropolitan' line. In what approximate year does it reach its absolute lowest point on the chart, and what is the approximate percent change at that time?",
        'Compare the three lines over the entire 50-year period. Which demographic category exhibits the highest volatility (the most dramatic peaks and valleys), and which one appears to be the most stable?',
    ],
)


## Section 3: Multi-Round Video Understanding

This case mirrors the multi-round video example in the script. The notebook records the prompt sequence and each generated answer while keeping the video display lightweight for GitHub.


In [ ]:
video_answers = run_multiround_video_case(
    title='Example 6: Multi-Round Video Understanding',
    video_name='video-example.mp4',
    questions=[
        'please describe the video in details',
        'At what timestamps is the Summar Palace mentioned?',
        'At what timestamps is the CHANG AN AVENUE mentioned?',
        'At what timestamps is the THE FINANCIAL STREET FORUM mentioned?',
    ],
    fps=1,
    max_frames=180,
)


## Section 4: Mixed-Modal Prompting

This example combines a video and an image in one user turn, then asks the model to produce a single creative response.


In [ ]:
mixed_response = run_mixed_case(
        title='Example 7: Mixed Video + Image Story Generation',
        video_name='polar_bear.mp4',
        image_name='horse_poet.png',
        question='Write a fairy tale based on the video and the image below:\\nVideo\\n',
        fps=1,
        max_frames=180,
    )


## Section 5: Text-Only Baseline

A plain-text example is useful as a minimal baseline when you want to compare multimodal behavior against a standard chat prompt.


In [ ]:
text_response = run_text_case(
    title='Example 8: Plain Text Conversation',
    question='There are ten birds in a tree. If you shoot and kill one, how many are left?',
)


## Optional Appendix: Capture the Raw Script Output

The cells above are the notebook-friendly version of the examples. If you also want exact terminal parity with `python3 inference/example_penguinvl.py`, run the next cell. It will launch the script in a subprocess and print the raw stdout and stderr.

Note: this loads the model again, so it is slower and uses additional GPU memory while it runs.


In [ ]:
RUN_RAW_SCRIPT_APPENDIX = False

if RUN_RAW_SCRIPT_APPENDIX:
    import subprocess

    cmd = [
        'python3',
        'inference/example_penguinvl.py',
        '--model-path',
        MODEL_PATH,
    ]

    result = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        check=False,
    )

    print('Return code:', result.returncode)
    print('\n=== STDOUT ===\n')
    print(result.stdout)
    print('\n=== STDERR ===\n')
    print(result.stderr)
else:
    print('Skipping raw script appendix. Set RUN_RAW_SCRIPT_APPENDIX = True to enable it.')


## Optional: How to Execute This Notebook from the Terminal

From the repo root, you can either open it interactively:

```bash
jupyter lab
```

or execute it in-place so the outputs are saved for GitHub rendering:

```bash
jupyter nbconvert --to notebook --execute --inplace inference/notebooks/01_penguinvl_inference_recipes.ipynb
```

If you use a custom environment or launcher, make sure the notebook kernel points to the same environment that successfully runs `python inference/example_penguinvl.py`.
